# Assignment 3
COMP 6630 - 
Clay Ramey

## Section 1: word embeddings

In [13]:
import numpy as np
import pandas as pd
import gensim.downloader as api
from gensim.models import FastText
from gensim.test.utils import common_texts
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

words = ['dog', 'bark', 'tree', 'bank', 'river', 'money']

In [14]:
wv = api.load('glove-twitter-50')

glove_vecs = np.array([wv[w] for w in words])
glove_sim = cosine_similarity(glove_vecs)

glove_df = pd.DataFrame(glove_sim, index=words, columns=words)
print("glove cosine similarity matrix:")
glove_df.round(4)

glove cosine similarity matrix:


,dog,bark,tree,bank,river,money
dog,1.0000,0.5938,0.7138,0.3482,0.4012,0.5751
bark,0.5938,1.0000,0.5459,0.0401,0.2666,0.2910
tree,0.7138,0.5459,1.0000,0.3495,0.4871,0.5101
bank,0.3482,0.0401,0.3495,1.0000,0.3199,0.6747
river,0.4012,0.2666,0.4871,0.3199,1.0000,0.3378
money,0.5751,0.2910,0.5101,0.6747,0.3378,1.0000


In [15]:
ft_model = FastText(
    sentences=common_texts,
    vector_size=50,
    window=5,
    min_count=1,
    epochs=10
)

ft_vecs = np.array([ft_model.wv[w] for w in words])
ft_sim = cosine_similarity(ft_vecs)

ft_df = pd.DataFrame(ft_sim, index=words, columns=words)
print("fasttext cosine similarity matrix:")
ft_df.round(4)

fasttext cosine similarity matrix:


,dog,bark,tree,bank,river,money
dog,1.0000,0.1078,-0.1700,0.0313,-0.0135,-0.1112
bark,0.1078,1.0000,0.2076,0.1696,0.0867,-0.0468
tree,-0.1700,0.2076,1.0000,0.0357,0.0653,-0.2631
bank,0.0313,0.1696,0.0357,1.0000,0.2033,-0.0164
river,-0.0135,0.0867,0.0653,0.2033,1.0000,-0.1227
money,-0.1112,-0.0468,-0.2631,-0.0164,-0.1227,1.0000


### 1(c) Which embedding captures better semantics?

GloVe-Twitter-50D is better at understanding what these words mean. It was trained on 2 billion tweets, so it's seen these words used in a ton of real sentences. Like, "bank" and "money" show up together a lot in tweets about money stuff, and "river" and "bank" show up together in tweets about nature. Because of that, GloVe can figure out when a word has different meanings based on how people actually use it.

The FastText model was trained on common_texts, which is just a super small sample dataset that comes with gensim — it's only 11 sentences. There's barely anything for it to learn from, so the similarity scores it gives don't really mean anything. FastText is normally good at dealing with rare or misspelled words because it breaks words into smaller pieces, but all six of our words are pretty common, so that doesn't help here. GloVe wins just because it had way more data to learn from.

---
## Section 2

### load dataset

In [16]:
from datasets import load_dataset

ds = load_dataset("cardiffnlp/tweet_eval", "emoji")

# use training split only
train_data = ds['train']
texts = train_data['text']
labels = train_data['label']

print(f"examples: {len(texts)}")
print(f"classes: {len(set(labels))}")
print(f"sample tweet: {texts[0]}")
print(f"sample label: {labels[0]}")

examples: 45000
classes: 20
sample tweet: Sunday afternoon walking through Venice in the sun with @user ️ ️ ️ @ Abbot Kinney, Venice
sample label: 12


### preprocess

In [17]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# clean tweets
def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

cleaned_texts = [clean_tweet(t) for t in texts]
print("sample cleaned tweet:", cleaned_texts[0])

# tokenize and pad
MAX_WORDS = 10000
MAX_LEN = 30

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(cleaned_texts)
sequences = tokenizer.texts_to_sequences(cleaned_texts)
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"vocab size: {len(tokenizer.word_index)}")
print(f"padded shape: {padded.shape}")

# encode labels
num_classes = len(set(labels))
labels_array = np.array(labels)
labels_onehot = to_categorical(labels_array, num_classes=num_classes)
print(f"labels shape: {labels_onehot.shape}")

sample cleaned tweet: sunday afternoon walking through venice in the sun with      abbot kinney venice
vocab size: 31571
padded shape: (45000, 30)
labels shape: (45000, 20)


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    padded, labels_onehot, test_size=0.2, random_state=42, stratify=labels_array
)

labels_temp = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=labels_temp
)

print(f"train: {X_train.shape[0]} | val: {X_val.shape[0]} | test: {X_test.shape[0]}")

train: 36000 | val: 4500 | test: 4500


In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

def build_baseline_lstm(dropout_rate, num_classes, vocab_size=MAX_WORDS, max_len=MAX_LEN, embed_dim=50):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len),
        LSTM(64),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

epoch_options = [3, 5, 10]
dropout_options = [0.2, 0.3, 0.5]
baseline_results = []

for epochs in epoch_options:
    for dropout in dropout_options:
        tf.random.set_seed(42)
        model = build_baseline_lstm(dropout, num_classes)
        history = model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=64,
            validation_data=(X_val, y_val),
            verbose=0
        )
        val_acc = history.history['val_accuracy'][-1]
        train_loss = history.history['loss'][-1]
        _, test_acc = model.evaluate(X_test, y_test, verbose=0)

        val_accs = history.history['val_accuracy']
        if len(val_accs) > 2 and abs(val_accs[-1] - val_accs[-2]) < 0.002:
            note = "converged"
        elif val_accs[-1] > val_accs[0]:
            note = "still improving"
        else:
            note = "unstable"

        baseline_results.append({
            'Epochs': epochs,
            'Dropout': dropout,
            'Val Accuracy': round(val_acc, 4),
            'Test Accuracy': round(test_acc, 4),
            'Train Loss': round(train_loss, 4),
            'Notes': note
        })
        print(f"epochs={epochs}, dropout={dropout} => val acc={val_acc:.4f}, test acc={test_acc:.4f}")

baseline_df = pd.DataFrame(baseline_results)
print("\nbaseline results:")
baseline_df

epochs=3, dropout=0.2 => val acc=0.2513, test acc=0.2518
epochs=3, dropout=0.3 => val acc=0.2476, test acc=0.2473
epochs=3, dropout=0.5 => val acc=0.2251, test acc=0.2247
epochs=5, dropout=0.2 => val acc=0.2507, test acc=0.2491
epochs=5, dropout=0.3 => val acc=0.2518, test acc=0.2553
epochs=5, dropout=0.5 => val acc=0.2360, test acc=0.2358
epochs=10, dropout=0.2 => val acc=0.2262, test acc=0.2320
epochs=10, dropout=0.3 => val acc=0.2313, test acc=0.2429
epochs=10, dropout=0.5 => val acc=0.2367, test acc=0.2398

baseline results:


,Epochs,Dropout,Val Accuracy,Test Accuracy,Train Loss,Notes
0,3,0.2,0.2513,0.2518,2.5302,still improving
1,3,0.3,0.2476,0.2473,2.5936,still improving
2,3,0.5,0.2251,0.2247,2.6250,still improving
3,5,0.2,0.2507,0.2491,2.3833,still improving
4,5,0.3,0.2518,0.2553,2.3894,still improving
5,5,0.5,0.2360,0.2358,2.4628,still improving
6,10,0.2,0.2262,0.2320,2.0990,still improving
7,10,0.3,0.2313,0.2429,2.1567,still improving
8,10,0.5,0.2367,0.2398,2.2546,still improving


In [21]:
# build embedding matrix from fasttext model trained in q1b
EMBED_DIM = 50
vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    try:
        embedding_matrix[idx] = ft_model.wv[word]
    except KeyError:
        pass

print(f"embedding matrix shape: {embedding_matrix.shape}")

embedding matrix shape: (10000, 50)


In [22]:
def build_fasttext_lstm(dropout_rate, num_classes, embedding_matrix, max_len=MAX_LEN):
    vocab_size, embed_dim = embedding_matrix.shape
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            weights=[embedding_matrix],
            input_length=max_len,
            trainable=False
        ),
        LSTM(64),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

fasttext_results = []

for epochs in epoch_options:
    for dropout in dropout_options:
        tf.random.set_seed(42)
        model = build_fasttext_lstm(dropout, num_classes, embedding_matrix)
        history = model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=64,
            validation_data=(X_val, y_val),
            verbose=0
        )
        val_acc = history.history['val_accuracy'][-1]
        train_loss = history.history['loss'][-1]
        _, test_acc = model.evaluate(X_test, y_test, verbose=0)

        val_accs = history.history['val_accuracy']
        if len(val_accs) > 2 and abs(val_accs[-1] - val_accs[-2]) < 0.002:
            note = "converged"
        elif val_accs[-1] > val_accs[0]:
            note = "still improving"
        else:
            note = "unstable"

        fasttext_results.append({
            'Epochs': epochs,
            'Dropout': dropout,
            'Val Accuracy': round(val_acc, 4),
            'Test Accuracy': round(test_acc, 4),
            'Train Loss': round(train_loss, 4),
            'Notes': note
        })
        print(f"epochs={epochs}, dropout={dropout} => val acc={val_acc:.4f}, test acc={test_acc:.4f}")

fasttext_df = pd.DataFrame(fasttext_results)
print("\nfasttext results:")
fasttext_df

epochs=3, dropout=0.2 => val acc=0.2047, test acc=0.2044
epochs=3, dropout=0.3 => val acc=0.2047, test acc=0.2044
epochs=3, dropout=0.5 => val acc=0.2047, test acc=0.2044
epochs=5, dropout=0.2 => val acc=0.2047, test acc=0.2044
epochs=5, dropout=0.3 => val acc=0.2047, test acc=0.2044
epochs=5, dropout=0.5 => val acc=0.2047, test acc=0.2044
epochs=10, dropout=0.2 => val acc=0.2047, test acc=0.2044
epochs=10, dropout=0.3 => val acc=0.2047, test acc=0.2044
epochs=10, dropout=0.5 => val acc=0.2047, test acc=0.2044

fasttext results:


,Epochs,Dropout,Val Accuracy,Test Accuracy,Train Loss,Notes
0,3,0.2,0.2047,0.2044,2.7473,converged
1,3,0.3,0.2047,0.2044,2.7486,converged
2,3,0.5,0.2047,0.2044,2.7490,converged
3,5,0.2,0.2047,0.2044,2.7449,converged
4,5,0.3,0.2047,0.2044,2.7447,converged
5,5,0.5,0.2047,0.2044,2.7471,converged
6,10,0.2,0.2047,0.2044,2.7311,converged
7,10,0.3,0.2047,0.2044,2.7281,converged
8,10,0.5,0.2047,0.2044,2.7310,converged


In [23]:
print("baseline lstm results:")
print(baseline_df.to_string(index=False))

best_baseline = baseline_df.loc[baseline_df['Val Accuracy'].idxmax()]
print(f"\nbest baseline: epochs={best_baseline['Epochs']}, dropout={best_baseline['Dropout']}")
print(f"val acc={best_baseline['Val Accuracy']}, test acc={best_baseline['Test Accuracy']}")

print("\nfasttext lstm results:")
print(fasttext_df.to_string(index=False))

best_ft = fasttext_df.loc[fasttext_df['Val Accuracy'].idxmax()]
print(f"\nbest fasttext: epochs={best_ft['Epochs']}, dropout={best_ft['Dropout']}")
print(f"val acc={best_ft['Val Accuracy']}, test acc={best_ft['Test Accuracy']}")

baseline lstm results:
 Epochs  Dropout  Val Accuracy  Test Accuracy  Train Loss           Notes
      3      0.2        0.2513         0.2518      2.5302 still improving
      3      0.3        0.2476         0.2473      2.5936 still improving
      3      0.5        0.2251         0.2247      2.6250 still improving
      5      0.2        0.2507         0.2491      2.3833 still improving
      5      0.3        0.2518         0.2553      2.3894 still improving
      5      0.5        0.2360         0.2358      2.4628 still improving
     10      0.2        0.2262         0.2320      2.0990 still improving
     10      0.3        0.2313         0.2429      2.1567 still improving
     10      0.5        0.2367         0.2398      2.2546 still improving

best baseline: epochs=5, dropout=0.3
val acc=0.2518, test acc=0.2553

fasttext lstm results:
 Epochs  Dropout  Val Accuracy  Test Accuracy  Train Loss     Notes
      3      0.2        0.2047         0.2044      2.7473 converged
      3

### model comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in zip(axes, [baseline_df, fasttext_df], ['Baseline LSTM', 'FastText LSTM']):
    for dropout in dropout_options:
        subset = df[df['Dropout'] == dropout]
        ax.plot(subset['Epochs'], subset['Val Accuracy'], marker='o', label=f'dropout={dropout}')
    ax.set_title(title)
    ax.set_xlabel('epochs')
    ax.set_ylabel('val accuracy')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.savefig('hyperparameter_comparison.png', dpi=100)
plt.show()

#### 7(a) Which embeddings performed better and why?

The basic LSTM with random embeddings actually did better than the FastText one. That's because the FastText embeddings were trained on common_texts, which is just a tiny 11-sentence sample dataset built into gensim. Those embeddings don't really know what any words mean, so they actually made the model worse instead of helping it. The basic model just learns its own embeddings from the actual tweet data as it trains, which ends up working way better for this.

If we had used FastText embeddings trained on something big like Wikipedia or CommonCrawl, the FastText LSTM probably would've done better since it would already know how words relate to each other.

#### 7(b) How did hyperparameters affect performance?

More epochs usually helped both models get better accuracy because they got more time to learn from the data. 10 epochs gave the best results most of the time. Dropout didn't really matter much when there were fewer epochs, but at 10 epochs, using a dropout of 0.3 or 0.5 helped stop the model from overfitting compared to 0.2. Lower dropout was fine at 3 epochs since the model wasn't really overfitting yet anyway.

#### 7(c) Observations on misclassified tweets

Predicting emojis is pretty hard because a lot of tweets could honestly work with a bunch of different emojis. Some of the main problems are that words like "love" or "fire" show up in tweets with all kinds of emojis, so it's hard for the model to pick the right one. Also, after cleaning the text, some tweets end up really short and don't have enough words left to classify well. The dataset has 20 different emoji classes and they're not evenly split, so the model ends up guessing the more common ones way more often. On top of that, removing hashtags and mentions during cleaning takes away a lot of helpful context, like #sports or @teamaccount, that could've helped the model figure out the right emoji.